# 🚀 Nemotron-3-Nano Fine-tuning with NeMo AutoModel (BIRD SQL)

This notebook walks you through **PEFT (LoRA) fine-tuning** of the NVIDIA Nemotron-3-Nano-30B model using the **NeMo AutoModel** framework and the **BIRD SQL** text-to-SQL dataset. Training runs via a YAML config and produces checkpoints; Step 5 times base-only inference (sanity check); Step 6 loads the base model and trained PEFT adapter for **local inference** (no NIM); Step 7 runs a lightweight BIRD-style evaluation comparing **base + PEFT** vs **base only** on the same dev set.

---

## 📋 What You're Working With

| | |
|:--:|:--|
| 🤖 **Model** | `NVIDIA-Nemotron-3-Nano-30B-A3B-BF16` |
| 🛠️ **Framework** | NeMo AutoModel (Hugging Face–native) |
| 📐 **Method** | LoRA (Parameter-Efficient Fine-Tuning) |
| 📊 **Dataset** | BIRD SQL (official: `birdsql/bird23-train-filtered`) |

---

## ✅ Prerequisites

### 💻 Hardware
- **2× GPUs** — NVIDIA H100, A100, or similar (this cookbook is set for 2 GPUs)
- Sufficient storage for model cache and checkpoints

### 📦 Software
- **OS:** Linux (e.g. Ubuntu 22.04)
- **Docker** and **NVIDIA Container Toolkit** (for GPU access in the container)
- Use the **NeMo AutoModel container** so PyTorch and Mamba (Nemotron-3-Nano) are ABI-matched—no local venv needed.
- **Jupyter:** After starting the container, register and select the **Python (AutoModel venv)** kernel so the notebook has `nemo_automodel` (Step 1).

---

## 🗺️ Workflow at a Glance

| Step | What you'll do |
|:--:|:--|
| **1** | 🐳 Pull and run the NeMo AutoModel container (mount this repo); register and select **Python (AutoModel venv)** kernel for Jupyter |
| **2** | 📊 Prepare BIRD SQL dataset (train → JSONL) |
| **3** | ⚙️ Configure PEFT training (YAML) |
| **4** | 🎯 Run fine-tuning inside the container |
| **5** | **Base-only inference timing** — load 30B base (no PEFT), run a few test queries, report time per query (sanity check before full eval) |
| **6** | Load base model + PEFT adapter for local inference (NeMo AutoModel preferred, HF fallback) |
| **7** | Lightweight BIRD-style eval: compare base+PEFT vs base-only on the same dev set |
| **6B** | Two-container setup: Docker network, merge adapter → merged model, spin up NIM container with merged model |
| **7B** | Eval via NIM: prove network, then run a 3-sample inference check (input vs gold vs model output) |

Follow the steps below in order. Let's go! 👇

---

## Step 1: Run with the NeMo AutoModel Container

To avoid PyTorch/Mamba ABI issues, run this cookbook **inside** the official NeMo AutoModel container. The image has NeMo AutoModel, PyTorch, and Mamba support built for Nemotron-3-Nano.

### Start Jupyter from the notebook directory (avoid 403 and missing files)

**Inside the container**, start Jupyter from the directory that contains this notebook so the file tree shows your repo and the correct token is used:

```bash
cd /workspace/usage-cookbook/Nemotron-3-Nano/finetuning_and_deployment
jupyter notebook --ip=0.0.0.0 --allow-root
```

- If you start from `/opt` or elsewhere, Jupyter will serve that directory and this notebook won't appear in the file tree.
- **403 Forbidden:** Your browser may have an old cookie for `localhost:8889`. Use the **full URL with token** printed when Jupyter starts (e.g. `http://localhost:8889/tree?token=...`), or open a new incognito/private window and paste that URL.
- The venv has no `pip` binary; use `/opt/venv/bin/python3 -m pip` if you need to install packages.

### Pull and start the container

Run these commands **on your host** (outside the container). Mount your repo so this directory is available at `/workspace` inside the container. Adjust `REPO_PATH` to the path that contains `usage-cookbook` (e.g. your Nemotron fork root).

### Connect Jupyter to the AutoModel venv kernel (do this once)

When you start Jupyter inside the container, the **default kernel** is often system Python, which does not have `nemo_automodel`. Register the container's venv so the notebook uses the right environment.

**Option A — In a terminal inside the container** (after `source /opt/venv/bin/activate`):
```bash
pip install ipykernel
python -m ipykernel install --user --name automodel-venv --display-name "Python (AutoModel venv)"
```
Then start (or restart) Jupyter. In the notebook: **Kernel → Change Kernel → "Python (AutoModel venv)"**.

**Option B — Run the cell below** from this notebook (any kernel). It registers the venv at `/opt/venv`. Then restart Jupyter and select **"Python (AutoModel venv)"** in the notebook.

In [1]:
# Register the AutoModel venv as a Jupyter kernel (run once per container).
import subprocess
import sys
venv_python = "/opt/venv/bin/python"
subprocess.check_call([venv_python, "-m", "pip", "install", "-q", "ipykernel"])
subprocess.check_call([venv_python, "-m", "ipykernel", "install", "--user", "--name", "automodel-venv", "--display-name", "Python (AutoModel venv)"])
print('Kernel registered. Restart Jupyter, then in the notebook choose: Kernel → Change Kernel → "Python (AutoModel venv)"')

FileNotFoundError: [Errno 2] No such file or directory: '/opt/venv/bin/python'

In [2]:
import sys
print(sys.executable)

/opt/venv/bin/python


In [1]:
# Run these on the HOST (terminal), not in the notebook, to start the container:
#   export REPO_PATH=/home/shadeform/30b-bird   # path to repo containing usage-cookbook
#   docker pull nvcr.io/nvidia/nemo-automodel:25.11.00
#   docker run -it --gpus all -p 8889:8888 -v "${REPO_PATH}:/workspace" nvcr.io/nvidia/nemo-automodel:25.11.00 bash
# Then inside the container:  cd /workspace/usage-cookbook/Nemotron-3-Nano/finetuning_and_deployment
# Start Jupyter from there if you want to run this notebook, or run the steps manually.

# Optional: if you're already inside the container, verify the environment:
import sys
try:
    import nemo_automodel
    import datasets
    print("nemo_automodel:", nemo_automodel.__file__)
    print("datasets OK")
except ImportError as e:
    print("ImportError (run this notebook inside the container):", e)
try:
    import mamba_ssm
    print("mamba_ssm OK")
except ImportError as e:
    print("mamba_ssm:", e)

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


nemo_automodel: /opt/Automodel/nemo_automodel/__init__.py
datasets OK
mamba_ssm OK


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


mamba_ssm OK


**If the cell above reports `No module named 'nemo_automodel'`:** Some variants of the NeMo AutoModel container do not pre-install the `nemo_automodel` pip package. Run the cell below **once** (inside this container's Jupyter) to install it, then re-run the verification cell.

In [6]:
# Run once if nemo_automodel is missing in this container. Then re-run the verification cell above.
import subprocess
import sys
result = subprocess.run([sys.executable, "-m", "pip", "install", "nemo_automodel", "datasets"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("stderr:", result.stderr)
else:
    print("Install finished. Re-run the verification cell above.")


Install finished. Re-run the verification cell above.


### Container image

- **Image:** `nvcr.io/nvidia/nemo-automodel:25.11.00`
- **Why:** Includes NeMo AutoModel, PyTorch, and Mamba support built together so Nemotron-3-Nano loads without ABI errors. No need to install `mamba-ssm` or `nemo_automodel` on the host.
- **For Steps 6B/7B (NIM eval):** Start the container with a Docker network so the notebook can reach NIM: add `--network evalnet --name notebook` to your `docker run` (and create the network once with `docker network create evalnet`). See Step 6B and TWO_CONTAINER_NOTEBOOK_AND_INFERENCE.md.

### Accessing Jupyter from your browser (e.g. on Brev / Cursor)

If you started the container with **port 8889** (e.g. `-p 8889:8888`) so the host's Jupyter can stay on 8888:

1. **Forward port 8889** in Cursor: open the **Ports** view (View → Ports, or the "Ports" tab in the bottom panel). Click **Forward a Port** and add **8889**.
2. In the container, Jupyter prints a URL with a **token** (e.g. `http://127.0.0.1:8888/tree?token=...`). Use that token with port **8889** in your browser: **http://localhost:8889/tree?token=YOUR_TOKEN** (or click the localhost:8889 link in the Ports view).
3. The container's Jupyter (8889) and the host's Jupyter (8888) do not interfere—they are separate ports.

## Step 2: Prepare the BIRD SQL Dataset

We use the **official** BIRD training set from the BIRD team on Hugging Face: **`birdsql/bird23-train-filtered`** (6,601 examples). Each row has `question`, `evidence`, `SQL`, and `db_id`. We build a single **input** (question + evidence) and **output** (SQL) per example and save **JSONL** for AutoModel.

In [8]:
import os
from datasets import load_dataset

DATASET_DIR = os.environ.get("DATASET_DIR", os.path.join(os.getcwd(), "dataset"))
os.makedirs(DATASET_DIR, exist_ok=True)
training_jsonl = os.path.join(DATASET_DIR, "training.jsonl")

def row_to_input_output(example):
    question = example.get("question", "")
    evidence = example.get("evidence") or ""
    sql = example.get("SQL", "")
    input_text = f"{question}\n{evidence}".strip()
    return {"input": input_text, "output": sql.strip()}

print("Loading birdsql/bird23-train-filtered (train split)...")
ds = load_dataset("birdsql/bird23-train-filtered", split="train")
ds = ds.map(row_to_input_output, remove_columns=ds.column_names, num_proc=4)
ds.to_json(training_jsonl, orient="records", lines=True, force_ascii=False)
print(f"Saved {len(ds)} examples to {training_jsonl}")

Loading birdsql/bird23-train-filtered (train split)...


Creating json from Arrow format: 100%|█████████████████████████████████████████████████| 7/7 [00:00<00:00, 276.90ba/s]

Saved 6601 examples to /workspace/usage-cookbook/Nemotron-3-Nano/finetuning_and_deployment/dataset/training.jsonl


## Step 3: PEFT Training Config (YAML)

Create a YAML config that points the AutoModel PEFT recipe at the Nemotron-3-Nano model and your BIRD JSONL. The dataset is loaded via **ColumnMappedTextInstructionDataset** with `input` → question and `output` → answer, and **answer-only loss**.

In [9]:
import os

DATASET_DIR = os.environ.get("DATASET_DIR", os.path.join(os.getcwd(), "dataset"))
training_jsonl = os.path.join(DATASET_DIR, "training.jsonl")
config_path = os.path.join(os.getcwd(), "bird_peft_nemotron_nano.yaml")

yaml_content = f"""# PEFT (LoRA) fine-tuning: Nemotron-3-Nano on BIRD SQL (2 GPUs)
# Run from this directory inside the container: torchrun --nproc-per-node=2 finetune.py

step_scheduler:
  global_batch_size: 16
  local_batch_size: 8
  ckpt_every_steps: 100
  val_every_steps: 50
  num_epochs: 1

dist_env:
  backend: nccl
  timeout_minutes: 60

rng:
  _target_: nemo_automodel.components.training.rng.StatefulRNG
  seed: 1111
  ranked: true

model:
  _target_: nemo_automodel.NeMoAutoModelForCausalLM.from_pretrained
  pretrained_model_name_or_path: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
  torch_dtype: bf16

checkpoint:
  enabled: true
  checkpoint_dir: checkpoints/
  model_save_format: safetensors
  save_consolidated: false

peft:
  _target_: nemo_automodel.components._peft.lora.PeftConfig
  match_all_linear: true
  dim: 8
  alpha: 32
  use_triton: true

distributed:
  _target_: nemo_automodel.components.distributed.fsdp2.FSDP2Manager
  dp_size: null
  tp_size: 1
  cp_size: 1
  sequence_parallel: false
  backend: nccl

loss_fn:
  _target_: nemo_automodel.components.loss.masked_ce.MaskedCrossEntropy

dataset:
  _target_: nemo_automodel.components.datasets.llm.column_mapped_text_instruction_dataset.ColumnMappedTextInstructionDataset
  path_or_dataset_id: dataset/training.jsonl
  # split is ignored for local JSONL; omit or set for HF datasets
  split: train
  column_mapping:
    question: input
    answer: output
  answer_only_loss_mask: true

packed_sequence:
  packed_sequence_size: 0

dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn: nemo_automodel.components.datasets.utils.default_collater
  shuffle: false

optimizer:
  _target_: torch.optim.Adam
  betas: [0.9, 0.999]
  eps: 1e-8
  lr: 1.0e-5
  weight_decay: 0
"""

with open(config_path, "w") as f:
    f.write(yaml_content)
print(f"Wrote config to {config_path}")
print(f"Dataset path in config: dataset/training.jsonl")

Wrote config to /workspace/usage-cookbook/Nemotron-3-Nano/finetuning_and_deployment/bird_peft_nemotron_nano.yaml
Dataset path in config: dataset/training.jsonl


**Note:** If your NeMo AutoModel recipe expects a different dataset API (e.g. no `split` for a single file), adjust the `dataset` block. For a single JSONL file, some setups use `path_or_dataset_id` only and no `split`; if you see errors, try removing the `split: train` line or set `split: null`.

## Step 4: Run Fine-Tuning

**Inside the container**, from this directory (`/workspace/usage-cookbook/Nemotron-3-Nano/finetuning_and_deployment`), run the PEFT recipe. The local `finetune.py` defaults to `bird_peft_nemotron_nano.yaml`. Use the container's Python (no `uv run`). Adjust `--nproc-per-node` to your GPU count.

In [10]:
%%bash
# Run from this directory inside the container. Local finetune.py uses bird_peft_nemotron_nano.yaml by default.
# This cookbook is set for 2 GPUs. Use the container's Python (no uv).

echo "Run fine-tuning (inside the container). Prepend 'time ' to measure training duration:"
echo "  time torchrun --nproc-per-node=2 finetune.py --config bird_peft_nemotron_nano.yaml"
echo "Or without timing:"
echo "  torchrun --nproc-per-node=2 finetune.py --config bird_peft_nemotron_nano.yaml"

Run fine-tuning (inside the container):
  torchrun --nproc-per-node=2 finetune.py
Or with explicit config:
  torchrun --nproc-per-node=2 finetune.py --config bird_peft_nemotron_nano.yaml


**Run setup (2 GPUs, &lt;2h):**
- **2 GPUs:** `torchrun --nproc-per-node=2` launches 2 workers; logs show "World size: 2". With `tp_size: 1`, both GPUs are used in data parallel (FSDP2). No change needed.
- **Training time:** Full BIRD train (6601 samples) with global_batch_size 24 ≈ 275 steps per epoch. To **measure elapsed time**, run: `time torchrun --nproc-per-node=2 finetune.py --config bird_peft_nemotron_nano.yaml` (shell prints real/user/sys when done).
- **Warning:** `The given buffer is not writable` from `hf_storage.py` when loading the model is a known PyTorch/safetensors quirk; it is suppressed after the first time and does not affect training or correctness. Safe to ignore.

**Key parameters (edit in YAML or override via CLI):**
- `step_scheduler.global_batch_size`, `local_batch_size`: batch sizes
- `step_scheduler.num_epochs`, `ckpt_every_steps`: training length and checkpoint frequency
- `peft.dim`, `peft.alpha`: LoRA rank and scaling
- `optimizer.lr`: learning rate

**Hardware:** This config uses **2 GPUs** (`--nproc-per-node=2`). Batch sizes are set for 2-way (global_batch_size 16, local_batch_size 8).

## Step 5: Base-only inference timing (sanity check before full eval)

Before loading PEFT or running full eval, **measure how long inference takes** with the 30B base model only. This step loads the base model, runs a small number of test queries with the same chat template as training, and reports **time per query** and **total time**. Use this to confirm inference speed is acceptable; if it's too slow, tune batch size or sequence length before running Step 7.

**Memory:** After timing, the cell frees the base model so Step 6 can load base+PEFT without holding two copies in memory.

**Attention fallback:** You may see "Falling back to sdpa / eager attention" and "Retrying without SDPA patching." This model (Mamba+transformer hybrid) often doesn't use optimized attention in the transformers/NeMo path; inference still works but is slower. For faster inference with this architecture, use **SGLang** (see the repo's sglang_cookbook) or serve via NVIDIA NIM.

**Speed vs length:** Lower `MAX_NEW_TOKENS` (e.g. 64 or 32) to make each query faster—the model generates fewer tokens before stopping. If answers get cut off, raise it again. The reported tok/s tells you generation throughput.

**Why ~1.5–2 tok/s?** In this notebook we use the transformers/NeMo path, which falls back to **eager attention** for this Mamba+transformer model—no SDPA/Flash. So decode is slow. To get **10+ tok/s**, use **SGLang** (see repo sglang_cookbook) or **NVIDIA NIM**; they use optimized kernels for this architecture. See SETUP_GOTCHAS.md "Inference speed" for more.

In [5]:
# Step 5: Base-only inference timing — load 30B base, run N test queries, report time
import os
import time
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
import torch

BASE_ID = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
NUM_TEST_QUERIES = 5   # number of test prompts for timing
MAX_NEW_TOKENS = 128   # lower = faster (fewer tokens to generate); raise if outputs get cut off

def _load_base():
    try:
        from nemo_automodel import NeMoAutoModelForCausalLM
        m = NeMoAutoModelForCausalLM.from_pretrained(
            BASE_ID,
            dtype=torch.bfloat16,
            trust_remote_code=True,
            device_map="auto",
        )
        return m, getattr(m, "tokenizer", None)
    except Exception as e:
        print(f"NeMo AutoModel load failed ({e}), falling back to HF transformers.")
        from transformers import AutoModelForCausalLM, AutoTokenizer
        m = AutoModelForCausalLM.from_pretrained(
            BASE_ID,
            dtype=torch.bfloat16,
            trust_remote_code=True,
            device_map="auto",
        )
        tok = AutoTokenizer.from_pretrained(BASE_ID, trust_remote_code=True)
        return m, tok

# Simple test prompts (short; same style as BIRD input)
TEST_PROMPTS = [
    "Given the following schema, write a SQL query: table users (id int, name text). Question: List all users.",
    "Schema: orders(id int, user_id int, amount float). Question: Total amount per user.",
    "Tables: products(id, name), sales(product_id, qty). Question: Top 5 products by quantity sold.",
    "Schema: logs(ts timestamp, level text). Question: Count errors in the last hour.",
    "Write SQL: employees(id, dept_id), departments(id, name). Question: Employee count per department.",
]

print("Loading base model (no PEFT)...")
t0 = time.perf_counter()
base_model, tokenizer = _load_base()
if tokenizer is None:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(BASE_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
load_time = time.perf_counter() - t0
print(f"Model loaded in {load_time:.1f}s")

def build_prompt(text):
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": ""}, {"role": "user", "content": text}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

prompts = [build_prompt(p) for p in TEST_PROMPTS[:NUM_TEST_QUERIES]]
if tokenizer.pad_token is None or getattr(tokenizer, "pad_token_id", None) is None or tokenizer.pad_token_id < 0:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("=" * 60)
print("DEBUG: Test queries (full user text, before chat template)")
print("=" * 60)
for i, p in enumerate(TEST_PROMPTS[:NUM_TEST_QUERIES]):
    print(f"  [{i+1}] {p}")
print()

query_times = []
outputs_decoded = []
input_token_counts = []
output_token_counts = []
for i, (user_text, prompt) in enumerate(zip(TEST_PROMPTS[:NUM_TEST_QUERIES], prompts)):
    inputs_i = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs_i = {k: v.to(base_model.device) for k, v in inputs_i.items() if k in ("input_ids", "attention_mask")}
    n_input = inputs_i["input_ids"].shape[1]
    t0 = time.perf_counter()
    with torch.inference_mode():
        out_i = base_model.generate(
            **inputs_i,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.perf_counter() - t0
    gen_ids = out_i[0][n_input:]
    n_output = gen_ids.shape[0]
    query_times.append(elapsed)
    input_token_counts.append(n_input)
    output_token_counts.append(n_output)
    tokens_per_sec = n_output / elapsed if elapsed > 0 else 0
    text_out = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    outputs_decoded.append(text_out)
    hit_limit = n_output >= MAX_NEW_TOKENS
    print(f"--- Query {i+1}/{len(prompts)} ({elapsed:.2f}s) ---")
    print(f"  Response tokens: {n_output} / {MAX_NEW_TOKENS} max  |  tok/s: {tokens_per_sec:.1f}" + ("  [may be truncated]" if hit_limit else ""))
    print(f"  Input tokens: {n_input}")
    print(f"  Input (full): {user_text}")
    print(f"  Output (full): {text_out}")
    print()

total_time = sum(query_times)
total_out_tokens = sum(output_token_counts)
per_query = total_time / len(query_times)
avg_tok_per_sec = total_out_tokens / total_time if total_time > 0 else 0
print("=" * 60)
print(f"Total time: {total_time:.2f}s  |  Per query: {per_query:.2f}s  |  Queries: {len(prompts)}")
print(f"Total input tokens: {sum(input_token_counts)}  |  Total response tokens: {total_out_tokens}  |  Avg tok/s: {avg_tok_per_sec:.1f}")
print(f"Response tokens per query: {output_token_counts}  (max_new_tokens={MAX_NEW_TOKENS}; hit limit = likely truncated)")
print(f"Per-query times (s): {[round(t, 2) for t in query_times]}")
print(f"Per-query tok/s: {[round(o/t, 1) if t > 0 else 0 for o, t in zip(output_token_counts, query_times)]}")
print("=" * 60)

# Keep last output for cleanup (out_i is last); we'll del base_model and gen outputs
out = out_i

# Free memory for Step 6 (load base+PEFT without holding two copies)
del base_model
del out
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Base model freed. Proceed to Step 6 to load base+PEFT for eval.")

Loading base model (no PEFT)...


Loading checkpoint shards: 100%|████████| 13/13 [00:11<00:00,  1.11it/s]


Model loaded in 32.9s
DEBUG: Test queries (full user text, before chat template)
  [1] Given the following schema, write a SQL query: table users (id int, name text). Question: List all users.
  [2] Schema: orders(id int, user_id int, amount float). Question: Total amount per user.
  [3] Tables: products(id, name), sales(product_id, qty). Question: Top 5 products by quantity sold.
  [4] Schema: logs(ts timestamp, level text). Question: Count errors in the last hour.
  [5] Write SQL: employees(id, dept_id), departments(id, name). Question: Employee count per department.

--- Query 1/5 (49.14s) ---
  Response tokens: 66 / 128 max  |  tok/s: 1.3
  Input tokens: 41
  Input (full): Given the following schema, write a SQL query: table users (id int, name text). Question: List all users.
  Output (full): **SQL query**

```sql
SELECT id, name
FROM users;
```

This statement retrieves every row from the `users` table, returning the `id` and `name` columns for all users. If you simply want to li

## Step 6: Load Base Model and PEFT Adapter for Local Inference

Load the **base model** once and the **trained PEFT (LoRA) adapter** for local inference (no NIM, single container). We prefer **NeMo AutoModel** (same cache as training); if that fails, we fall back to Hugging Face `transformers`. We do **not** load a second copy of the base; Step 7 runs base-only eval by temporarily **disabling** the adapter on the same model, which keeps GPU memory use much lower and speeds up eval.

Checkpoint discovery: latest `epoch_*_step_*` under `checkpoint_dir` (e.g. `checkpoints/` or `checkpoints_epoch0_step83/`). Adapter must be in `<checkpoint>/model/` with `adapter_config.json` (and weights).

**Download progress:** The "Fetching N files" bar can stay at 0% for a long time when downloading large shards. For **incremental** (per-file or per-byte) progress, install and enable the Rust backend: `pip install hf_transfer` — the cell sets `HF_HUB_ENABLE_HF_TRANSFER=1` so it is used automatically when installed.

In [4]:
# Step 6: Load base model and PEFT adapter (NeMo AutoModel preferred, HF fallback) for local inference
# Enable hf_transfer if installed: shows incremental download progress instead of stuck at 0%% until each file finishes
import os
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
import torch
from peft import PeftModel

checkpoint_dir = "checkpoints"  # or checkpoints_epoch0_step83 if you use that name
base_id = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

# Find latest epoch_*_step_* checkpoint
if not os.path.isdir(checkpoint_dir):
    checkpoint_dir = "checkpoints_epoch0_step83"  # fallback
step_dirs = [d for d in os.listdir(checkpoint_dir) if os.path.isdir(os.path.join(checkpoint_dir, d)) and d.startswith("epoch_")]
def _step_num(name):
    parts = name.split("_")
    return int(parts[-1]) if len(parts) >= 4 and parts[-1].isdigit() else 0
step_dirs.sort(key=_step_num)
latest = step_dirs[-1] if step_dirs else None
if not latest:
    raise SystemExit(f"No epoch_*_step_* checkpoints in {checkpoint_dir}/")
adapter_path = os.path.join(checkpoint_dir, latest, "model")
print(f"Using adapter: {adapter_path}")

# Load base with NeMo AutoModel first (same cache as training); fallback to HF transformers
def _load_base():
    try:
        from nemo_automodel import NeMoAutoModelForCausalLM
        m = NeMoAutoModelForCausalLM.from_pretrained(
            base_id,
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
            device_map="auto",
        )
        return m, getattr(m, "tokenizer", None)
    except Exception as e:
        print(f"NeMo AutoModel load failed ({e}), falling back to HF transformers.")
        from transformers import AutoModelForCausalLM
        m = AutoModelForCausalLM.from_pretrained(
            base_id,
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
            device_map="auto",
        )
        return m, None

print("Loading base model (for PEFT)...")
base_model, _tok = _load_base()
if _tok is not None:
    tokenizer = _tok
else:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)

print("Loading PEFT adapter on top of base...")
model_peft = PeftModel.from_pretrained(base_model, adapter_path)

# We do NOT load a second base model (base_only). Step 7 will run base-only eval by
# temporarily disabling the adapter on model_peft, then re-enabling it. This cuts
# GPU memory in half so eval can run faster.
print("model_peft ready for Step 7 (base-only eval uses same model with adapter disabled).")

Using adapter: checkpoints/epoch_0_step_274/model
Loading base model (for PEFT)...


`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 13 files: 100%|██████████████████████████████████████| 13/13 [04:41<00:00, 21.62s/it]
Loading checkpoint shar


Loading PEFT adapter on top of base...
Loading base-only model (for comparison eval)...


Loading check


base_only and model_peft are ready for Step 6 eval.


## Step 7: Lightweight Eval — Base+PEFT vs Base Only

Load the eval set (from **dataset/eval_mini_dev.jsonl** or the same fallbacks as before), build the prompt with the **same chat template** as training (Nemotron: system + user, `add_generation_prompt=True`, `enable_thinking=False`), run **generate** for each example on **base+PEFT**, then on **base only** (same model, adapter disabled), then compare **exact match** and **normalized match** vs gold SQL.

**Making eval faster (360° view):**
- **Real-time progress:** Each sample prints current eval index and running exact/normalized accuracy so you see progress every step.
- **Software:** `EVAL_BATCH_SIZE` 2 or 4 runs multiple prompts per `generate()` (faster GPU use). `MAX_NEW_TOKENS=256` is usually enough for SQL and cuts time vs 512.
- **Hardware / 2 GPUs:** Right now both models share the same device(s) (e.g. `device_map="auto"`). To use 2 GPUs in parallel you’d load one model on `cuda:0` and the other on `cuda:1` and run the two evals in separate threads; that requires more VRAM (two full models). Alternatively run only one model at a time and use both GPUs for that model via `device_map="auto"` (already the case).
- **Common pitfalls:** (1) Eval set too large — use `EVAL_LIMIT=10` for a quick sanity check. (2) No batching — set `EVAL_BATCH_SIZE=2` or 4. (3) Very long `MAX_NEW_TOKENS` — reduce if SQL is short. (4) Running in a notebook with no progress — we now print per sample.

In [ ]:
# Step 7: Local eval — base+PEFT vs base-only (same eval data, same metrics)
import json
import os
import re
import torch

EVAL_LIMIT = 50  # cap examples (e.g. 10 quick, 50 or 500 full)
MAX_NEW_TOKENS = 256  # SQL usually <200 tokens; increase to 512 if you see truncation
EVAL_BATCH_SIZE = 1  # 2 or 4 can speed up (batched generate); 1 = simplest, real-time per sample

def load_eval_data(limit=50):
    """Load BIRD-style eval set: prefer dataset/eval_mini_dev.jsonl, else HF bird_mini_dev, else training.jsonl."""
    eval_path = "dataset/eval_mini_dev.jsonl"
    if os.path.exists(eval_path):
        out = []
        with open(eval_path) as f:
            for line in f:
                if limit and len(out) >= limit:
                    break
                row = json.loads(line)
                out.append({"input": row["input"], "output": row["output"]})
        if out:
            return out
    try:
        from datasets import load_dataset
        ds = load_dataset("birdsql/bird_mini_dev", split="mini_dev_sqlite")
        n = min(limit, len(ds)) if limit else len(ds)
        out = []
        for i in range(n):
            r = ds[i]
            question = r.get("question", "")
            evidence = r.get("evidence", "")
            schema = r.get("schema", "")
            sql = r.get("SQL", r.get("sql", "")).strip()
            user_input = "\n".join(filter(None, [schema, question, evidence])).strip()
            out.append({"input": user_input, "output": sql})
        return out
    except Exception as e:
        print(f"bird_mini_dev fallback failed: {e}")
    out = []
    path = "dataset/training.jsonl"
    if os.path.exists(path):
        with open(path) as f:
            for line in list(f)[-limit:]:
                row = json.loads(line)
                out.append({"input": row["input"], "output": row["output"]})
    return out

def build_prompt(tokenizer, user_text):
    """Same format as training: Nemotron chat template, no reasoning."""
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": ""},
            {"role": "user", "content": user_text.strip()},
        ],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    return prompt

def extract_sql_from_reply(reply):
    reply = (reply or "").strip()
    for start in ("SELECT", "select", "INSERT", "UPDATE", "DELETE", "WITH"):
        i = reply.upper().find(start.upper())
        if i != -1:
            return reply[i:].strip()
    return reply

def normalize_sql(s):
    if not s or not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def run_eval(model, tokenizer, eval_data, model_name="model", batch_size=1):
    exact_ok = norm_ok = 0
    n = len(eval_data)
    device = next(model.parameters()).device
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    for start in range(0, n, batch_size):
        batch_ex = eval_data[start:start + batch_size]
        prompts = [build_prompt(tokenizer, ex["input"]) for ex in batch_ex]
        if batch_size == 1:
            inputs = tokenizer(prompts[0], return_tensors="pt").to(device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=pad_id)
            reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            preds = [extract_sql_from_reply(reply)]
        else:
            inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=tokenizer.model_max_length or 4096).to(device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=pad_id)
            preds = []
            for b in range(len(batch_ex)):
                prompt_len = inputs["attention_mask"][b].sum().item()
                reply = tokenizer.decode(out[b][prompt_len:], skip_special_tokens=True)
                preds.append(extract_sql_from_reply(reply))
        for b, ex in enumerate(batch_ex):
            pred, gold = preds[b], ex["output"]
            if pred.strip() == gold.strip():
                exact_ok += 1
            if normalize_sql(pred) == normalize_sql(gold):
                norm_ok += 1
            i = start + b + 1
            exact_pct = round(100.0 * exact_ok / i, 1)
            norm_pct = round(100.0 * norm_ok / i, 1)
            print(f"  [{model_name}] Eval {i}/{n} | exact: {exact_pct}% | normalized: {norm_pct}%", flush=True)
    return {"n": n, "exact_match_pct": round(100.0 * exact_ok / n, 2) if n else 0, "normalized_match_pct": round(100.0 * norm_ok / n, 2) if n else 0}

eval_data = load_eval_data(limit=EVAL_LIMIT)
print(f"Eval set size: {len(eval_data)} (EVAL_LIMIT={EVAL_LIMIT}, MAX_NEW_TOKENS={MAX_NEW_TOKENS}, EVAL_BATCH_SIZE={EVAL_BATCH_SIZE})")

print("Running eval on base + PEFT...")
results_peft = run_eval(model_peft, tokenizer, eval_data, model_name="Base+PEFT", batch_size=EVAL_BATCH_SIZE)
print("Running eval on base only (adapter disabled)...")
model_peft.disable_adapter()
results_base = run_eval(model_peft, tokenizer, eval_data, model_name="Base only", batch_size=EVAL_BATCH_SIZE)
model_peft.enable_adapter()

print("\n--- Comparison ---")
print("Metric                     Base+PEFT          Base only")
print("-" * 60)
print(f"Exact match %             {results_peft['exact_match_pct']:<18} {results_base['exact_match_pct']}")
print(f"Normalized match %        {results_peft['normalized_match_pct']:<18} {results_base['normalized_match_pct']}")
print(f"N samples                 {results_peft['n']:<18} {results_base['n']}")


Eval set size: 50 (EVAL_LIMIT=50, MAX_NEW_TOKENS=256, EVAL_BATCH_SIZE=1)
Running eval on base + PEFT...
  [Base+PEFT] Eval 1/50 | exact: 0.0% | normalized: 0.0%


## Step 6B: Two-Container Setup — Docker Network and NIM with Merged Model

This step prepares **inference in a separate NIM container** so the notebook (this container) can run eval by calling the NIM API over a Docker network. No model is loaded in the notebook for 6B/7B; NIM serves the **merged** model (base + PEFT weights merged into one artifact).

**Flow (do in order):**

1. **Create Docker network** — On the host, create `evalnet` once (see commands in the cells below).
2. **Add this notebook container to evalnet** — So this notebook can reach NIM by hostname `nim`. If you did **not** start this container with `--network evalnet`, run **on the host**: `docker network connect evalnet <notebook_container_name>`. Get the container name with `docker ps`. Run the cell below to check; if it reports "not on evalnet", it will print the exact command. (If you start the container fresh next time, you can use `--network evalnet --name notebook` so this step isn't needed.)
3. **Merged model** — NIM for Nemotron-3-Nano serves the **merged** model only (no LoRA adapters in this image). If you don't have `merged_model/` yet, merge the adapter into the base (e.g. using the merge recipe or export step from the cookbook) and put the result in `merged_model/` in this directory.
4. **Start NIM** — In a **second terminal on the host**, run the NIM container on the same network. The cell below prints copy-paste commands for **merged model** (mount your fine-tuned artifact) or **base model** (NGC download, no export needed).

**Deploying NIM with the base model (no fine-tuning):** To run NIM with the **stock base model** (e.g. for a sanity check or comparison), you do **not** need to export the base locally. Use NIM's built-in download: do **not** set `NIM_DISABLE_MODEL_DOWNLOAD=1` and do **not** mount a model at `/opt/nim/workspace`. On first start the container will download and cache the default Nemotron-3-Nano model from NGC. The cell below prints commands for both **merged model (mount)** and **base model (NGC download)** — copy-paste the block you need.

After NIM is up, go to **Step 7B** to (1) send a basic request to NIM from this notebook to prove the network works, then (2) run full eval against the NIM endpoint.

In [13]:
# Step 6B — Check if this notebook container is on evalnet (so it can reach NIM at hostname 'nim').
import socket
import sys

def can_resolve_nim():
    try:
        socket.getaddrinfo("nim", 8000)
        return True
    except socket.gaierror:
        return False

on_evalnet = can_resolve_nim()
hostname = socket.gethostname()

if on_evalnet:
    print("✓ This container is on evalnet; hostname 'nim' will resolve once the NIM container is running.")
else:
    print("✗ This container is not on evalnet — hostname 'nim' cannot be resolved from here.")
    print()
    print("Run the following on the host (in a terminal outside the container):")
    print()
    print("  docker network connect evalnet <notebook_container_name>")
    print()
    print("To get the container name:  docker ps")
    print("This container's hostname (often the short container ID):", hostname)
    print("  → You can try:  docker network connect evalnet", hostname)
    print()
    print("Then re-run this cell to confirm. No need to restart Jupyter.")
    sys.exit(1)

✓ This container is on evalnet; hostname 'nim' will resolve once the NIM container is running.


In [14]:
# Step 6B: Ensure merged_model dir exists (for merged option) and print NIM run commands for the host.
import os

MERGED_DIR = "merged_model"
REPO_PATH = os.environ.get("REPO_PATH", "/path/to/30b-bird")

# Merged model: only warn if user will use that block
if not os.path.isdir(MERGED_DIR):
    print(f"⚠️  '{MERGED_DIR}/' not found.")
    print("   Create it by merging the PEFT adapter into the base model (e.g. use the merge/export step in this cookbook), then re-run this cell.")
    print("   You can still use the **base model (NGC download)** commands below — no export needed.\n")
else:
    print(f"✓ Found {MERGED_DIR}/ — ready for NIM (merged model).\n")

print("--- Copy-paste these commands in a **second terminal on the host** ---\n")
print("# 1. Create network (if not already created)")
print("docker network create evalnet 2>/dev/null || true\n")

# --- Option A: Merged (fine-tuned) model — mount local dir, disable NIM download ---
print("# --- Option A: Merged (fine-tuned) model — mount merged_model, NIM uses it only ---")
print(f'export REPO_PATH={REPO_PATH}')
print('export MERGED="${REPO_PATH}/usage-cookbook/Nemotron-3-Nano/finetuning_and_deployment/merged_model"')
print('docker run -it --rm --gpus all --network evalnet --name nim \\')
print('  -e NGC_API_KEY=$NGC_API_KEY \\')
print('  -e NIM_DISABLE_MODEL_DOWNLOAD=1 \\')
print('  -e NIM_RELAX_MEM_CONSTRAINTS=1 \\')
print('  -e NIM_MODEL_PROFILE=7cbe1181600064c6e10ebaf843497acae35aacff2ab96fe8247ae541ae0ac28a \\')
print('  -v "${MERGED}:/opt/nim/workspace" \\')
print('  -p 8000:8000 \\')
print('  nvcr.io/nim/nvidia/nemotron-3-nano:1.7.0-variant\n')

# --- Option B: Base model — NGC download, no mount ---
print("# --- Option B: Base model (NGC download) — no mount; NIM downloads and caches on first start ---")
print("#    Uses --name nim-base and -p 8001:8000 so host ports don't conflict if you have 4+ GPUs.")
print("#    With only 2 GPUs: run ONE NIM at a time (each needs both GPUs). Stop 'nim' before starting nim-base, or vice versa.")
print('docker run -it --rm --gpus all --network evalnet --name nim-base \\')
print('  -e NGC_API_KEY=$NGC_API_KEY \\')
print('  -e NIM_RELAX_MEM_CONSTRAINTS=1 \\')
print('  -e NIM_MODEL_PROFILE=7cbe1181600064c6e10ebaf843497acae35aacff2ab96fe8247ae541ae0ac28a \\')
print('  -p 8001:8000 \\')
print('  nvcr.io/nim/nvidia/nemotron-3-nano:1.7.0-variant\n')

print("--- Wait until NIM logs show 'Application is ready to receive API requests', then run Step 7B ---")
print("--- Two GPUs only? Run one NIM at a time. Four+ GPUs? You can run both; use NIM_BASE_URL = 'http://nim:8000' or 'http://nim-base:8000' ---")


✓ Found merged_model/ — ready for NIM (merged model).

--- Copy-paste these commands in a **second terminal on the host** ---

# 1. Create network (if not already created)
docker network create evalnet 2>/dev/null || true

# --- Option A: Merged (fine-tuned) model — mount merged_model, NIM uses it only ---
export REPO_PATH=/path/to/30b-bird
export MERGED="${REPO_PATH}/usage-cookbook/Nemotron-3-Nano/finetuning_and_deployment/merged_model"
docker run -it --rm --gpus all --network evalnet --name nim \
  -e NGC_API_KEY=$NGC_API_KEY \
  -e NIM_DISABLE_MODEL_DOWNLOAD=1 \
  -e NIM_RELAX_MEM_CONSTRAINTS=1 \
  -e NIM_MODEL_PROFILE=7cbe1181600064c6e10ebaf843497acae35aacff2ab96fe8247ae541ae0ac28a \
  -v "${MERGED}:/opt/nim/workspace" \
  -p 8000:8000 \
  nvcr.io/nim/nvidia/nemotron-3-nano:1.7.0-variant

# --- Option B: Base model (NGC download) — no mount; NIM downloads and caches on first start ---
docker run -it --rm --gpus all --network evalnet --name nim \
  -e NGC_API_KEY=$NGC_API_KEY \
  -e

### Step 7B.1a — Debug NIM inference (expected vs output)

Run this cell to inspect the **first few** NIM responses vs gold SQL. It prints, for each example:

- **INPUT** — First 250 chars of the prompt sent to NIM  
- **GOLD** — Expected SQL (from the eval set)  
- **RAW NIM content** — Exactly what the API returned (first 600 chars)  
- **EXTRACTED pred** — What we parse as SQL (from `extract_sql_from_reply`)  
- **finish_reason** — `"length"` means the reply was truncated; increase `MAX_NEW_TOKENS` in the cell below if you see this.  
- **Exact / Normalized match** — Whether the extracted SQL matches gold  

Use this to spot **truncation**, **thinking blocks** (e.g. `<think>...</think>`) in the reply, or wrong format.

## Step 7B: Eval via NIM — Prove Network, Debug, Then Full Eval

The NIM container should be **up and running** on the same Docker network as this notebook container (see Step 6B). We use the container name **`nim`** as the hostname so this notebook can reach NIM at `http://nim:8000`.

**7B.1 — Basic response**  
The first cell below sends a single chat request to the NIM endpoint from this notebook. If you get a valid JSON response with generated text, the Docker network is working and the notebook can talk to NIM.

**7B.1a — Debug inference**  
Run the debug cell to inspect the first few NIM responses (input, gold SQL, raw reply, extracted SQL, finish_reason). Use it to find truncation, thinking blocks, or format issues before running the full eval.

**7B.2 — Full eval**  
The last cell runs the same BIRD-style eval as Step 7 (exact match, normalized match) but sends each prompt to the NIM API and collects the model's reply, then compares to gold SQL.

**How Nemotron-3-Nano NIM inference works**  
Nemotron-3-Nano is a *reasoning* model: by default it generates chain-of-thought (thinking) first, then a final answer. NIM returns:
- **`message.content`** — The main answer (when reasoning is off or after thinking).
- **`message.reasoning` / `message.reasoning_content`** — The thinking trace when reasoning is on.
When reasoning is **on** (default), the model may put only the thinking in the response and leave `content` null, which breaks SQL extraction. For **text-to-SQL eval** we want the model to output the SQL directly in `content`. So we **disable reasoning** in the request using either:
1. **System prompt:** First message `{"role": "system", "content": "detailed thinking off"}`.
2. **Chat template kwargs:** `"chat_template_kwargs": {"enable_thinking": false}` in the request body.
We use both below and set `temperature=0` for deterministic SQL. To steer the model (base or fine-tuned) to output **only SQL** instead of prose, we set a **text-to-SQL system prompt** (`NIM_SYSTEM_FOR_SQL`) and append "Output only the SQL query, nothing else." to each user message. After that, the response should have `message.content` populated with the SQL (and we still fall back to `reasoning_content` in code if needed).

In [29]:
# Step 7B.1 — Prove Docker network: send one request to NIM from this notebook.
import requests

NIM_BASE_URL = "http://nim-base:8000"  # container name on the same Docker network
# Model ID must match what NIM serves (default: nvidia/nemotron-3-nano). If you set NIM_SERVED_MODEL_NAME when starting NIM, use that here.
NIM_MODEL_ID = "nvidia/nemotron-3-nano"
# System prompt for text-to-SQL eval (7B.1a, 7B.2): steers model to output only SQL, no prose.
NIM_SYSTEM_FOR_SQL = (
    "detailed thinking off. You are a text-to-SQL assistant. "
    "Given a question and evidence about the database, respond with only the SQL query, "
    "no explanation, no markdown, no other text."
)
url = f"{NIM_BASE_URL}/v1/chat/completions"

payload = {
    "model": NIM_MODEL_ID,
    "messages": [
        {"role": "system", "content": "detailed thinking off"},
        {"role": "user", "content": "Say hello in one word."},
    ],
    "max_tokens": 20,
    "temperature": 0,
    "chat_template_kwargs": {"enable_thinking": False},
}

try:
    r = requests.post(url, json=payload, timeout=30)
    r.raise_for_status()
    data = r.json()
    content = data.get("choices", [{}])[0].get("message", {}).get("content", "")
    print("✓ NIM responded. Content:", repr(content))
    print("  Docker network is working: this notebook can reach the NIM container.")
except requests.exceptions.ConnectionError as e:
    print("✗ Connection failed. Is NIM running on the same Docker network (evalnet)?")
    print("  From the host, start NIM with: --network evalnet --name nim")
    print("  Error:", e)
except Exception as e:
    print("Error:", e)

✓ NIM responded. Content: 'Hello'
  Docker network is working: this notebook can reach the NIM container.


In [30]:
# Step 7B.1a — Debug: inspect first N examples (expected vs NIM output).
# Requires: NIM_BASE_URL, NIM_MODEL_ID from Step 7B.1; same helpers as Step 7B.2.
import json
import os
import re
import requests

DEBUG_N = 3  # number of examples to print; increase if needed
MAX_NEW_TOKENS_DEBUG = 512  # enough for SQL; increase if still truncating

def load_eval_data_7b(limit=50):
    eval_path = "dataset/eval_mini_dev.jsonl"
    if os.path.exists(eval_path):
        out = []
        with open(eval_path) as f:
            for line in f:
                if limit and len(out) >= limit:
                    break
                row = json.loads(line)
                out.append({"input": row["input"], "output": row["output"]})
        if out:
            return out
    try:
        from datasets import load_dataset
        ds = load_dataset("birdsql/bird_mini_dev", split="mini_dev_sqlite")
        n = min(limit, len(ds)) if limit else len(ds)
        out = []
        for i in range(n):
            r = ds[i]
            question = r.get("question", "")
            evidence = r.get("evidence", "")
            schema = r.get("schema", "")
            sql = r.get("SQL", r.get("sql", "")).strip()
            user_input = "\n".join(filter(None, [schema, question, evidence])).strip()
            out.append({"input": user_input, "output": sql})
        return out
    except Exception as e:
        print(f"Fallback failed: {e}")
    return []

def extract_sql_from_reply(reply):
    reply = (reply or "").strip()
    for start in ("SELECT", "select", "INSERT", "UPDATE", "DELETE", "WITH"):
        i = reply.upper().find(start.upper())
        if i != -1:
            return reply[i:].strip()
    return reply

def normalize_sql(s):
    if not s or not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def get_content_from_nim_response(data):
    """Extract assistant text from NIM chat response. Handles content, content-as-list, and NIM reasoning/reasoning_content fields."""
    try:
        choice = data.get("choices", [{}])[0]
        msg = choice.get("message", {})
        content = msg.get("content")
        if isinstance(content, str) and content:
            return content
        if isinstance(content, list):
            parts = []
            for p in content:
                if isinstance(p, dict):
                    parts.append(p.get("text", p.get("content", "")))
                elif isinstance(p, str):
                    parts.append(p)
            return "\n".join(parts) if parts else ""
        # NIM Nemotron may put generated text in reasoning_content or reasoning when content is null
        rc = msg.get("reasoning_content") or msg.get("reasoning")
        if isinstance(rc, str) and rc.strip():
            return rc.strip()
        return ""
    except Exception:
        return ""

# Use NIM_BASE_URL and NIM_MODEL_ID from Step 7B.1 (run that cell first)
eval_data_debug = load_eval_data_7b(limit=DEBUG_N)
if not eval_data_debug:
    print("No eval data found. Create dataset/eval_mini_dev.jsonl or install datasets and use birdsql/bird_mini_dev.")
else:
    url = f"{NIM_BASE_URL}/v1/chat/completions"
    for i, ex in enumerate(eval_data_debug):
        payload = {
            "model": NIM_MODEL_ID,
            "messages": [
                {"role": "system", "content": NIM_SYSTEM_FOR_SQL},
                {"role": "user", "content": ex["input"].strip() + "\n\nOutput only the SQL query, nothing else."},
            ],
            "max_tokens": MAX_NEW_TOKENS_DEBUG,
            "temperature": 0,
            "chat_template_kwargs": {"enable_thinking": False},
        }
        try:
            r = requests.post(url, json=payload, timeout=120)
            r.raise_for_status()
            data = r.json()
            if i == 0:
                # Dump raw structure so we can see where the generated text lives
                c0 = data.get("choices", [{}])[0]
                print("RAW RESPONSE (first choice) keys:", list(c0.keys()) if isinstance(c0, dict) else type(c0))
                msg = c0.get("message", {})
                print("message keys:", list(msg.keys()) if isinstance(msg, dict) else type(msg))
                print("message.content type:", type(msg.get("content")))
                if msg.get("content") is not None:
                    print("message.content (first 300 chars):", repr(str(msg.get("content"))[:300]))
                print("Full first choice (truncated):")
                print(json.dumps(c0, indent=2, default=str)[:2000])
                print()
            content = get_content_from_nim_response(data)
            finish = data.get("choices", [{}])[0].get("finish_reason", "?")
        except Exception as e:
            content = ""
            finish = f"error: {e}"
        pred = extract_sql_from_reply(content)
        gold = ex["output"]
        print("=" * 60)
        print(f"Example {i+1}/{len(eval_data_debug)}")
        print("-" * 60)
        print("INPUT (first 250 chars):", repr((ex["input"].strip())[:250]))
        print("-" * 60)
        print("GOLD (expected SQL, first 400 chars):", repr(gold[:400]))
        print("-" * 60)
        print("RAW NIM content (first 600 chars):", repr(content[:600]))
        print("-" * 60)
        print("EXTRACTED pred (first 400 chars):", repr(pred[:400]))
        print("finish_reason:", finish)
        print("Exact match:", pred.strip() == gold.strip())
        print("Normalized match:", normalize_sql(pred) == normalize_sql(gold))
        print()

RAW RESPONSE (first choice) keys: ['index', 'message', 'logprobs', 'finish_reason', 'stop_reason', 'token_ids']
message keys: ['role', 'content', 'refusal', 'annotations', 'audio', 'function_call', 'tool_calls', 'reasoning', 'reasoning_content']
message.content type: <class 'str'>
message.content (first 300 chars): "SELECT COUNT(*) FILTER (WHERE Currency = 'EUR') * 1.0 / NULLIF(COUNT(*) FILTER (WHERE Currency = 'CZK'), 0) FROM customers;"
Full first choice (truncated):
{
  "index": 0,
  "message": {
    "role": "assistant",
    "content": "SELECT COUNT(*) FILTER (WHERE Currency = 'EUR') * 1.0 / NULLIF(COUNT(*) FILTER (WHERE Currency = 'CZK'), 0) FROM customers;",
    "refusal": null,
    "annotations": null,
    "audio": null,
    "function_call": null,
    "tool_calls": [],
    "reasoning": null,
    "reasoning_content": null
  },
  "logprobs": null,
  "finish_reason": "stop",
  "stop_reason": null,
  "token_ids": null
}

Example 1/3
---------------------------------------------------

In [28]:
# Step 7B.2 — Full BIRD eval against NIM endpoint.
import json
import os
import re
import requests

NIM_BASE_URL = "http://nim-base:8000"
NIM_MODEL_ID = "nvidia/nemotron-3-nano"  # must match NIM's served model (see Step 7B.1)
EVAL_LIMIT_7B = 50
MAX_NEW_TOKENS_7B = 512  # enough for SQL; increase if still truncating

def load_eval_data_7b(limit=50):
    eval_path = "dataset/eval_mini_dev.jsonl"
    if os.path.exists(eval_path):
        out = []
        with open(eval_path) as f:
            for line in f:
                if limit and len(out) >= limit:
                    break
                row = json.loads(line)
                out.append({"input": row["input"], "output": row["output"]})
        if out:
            return out
    try:
        from datasets import load_dataset
        ds = load_dataset("birdsql/bird_mini_dev", split="mini_dev_sqlite")
        n = min(limit, len(ds)) if limit else len(ds)
        out = []
        for i in range(n):
            r = ds[i]
            question = r.get("question", "")
            evidence = r.get("evidence", "")
            schema = r.get("schema", "")
            sql = r.get("SQL", r.get("sql", "")).strip()
            user_input = "\n".join(filter(None, [schema, question, evidence])).strip()
            out.append({"input": user_input, "output": sql})
        return out
    except Exception as e:
        print(f"Fallback failed: {e}")
    out = []
    if os.path.exists("dataset/training.jsonl"):
        with open("dataset/training.jsonl") as f:
            for line in list(f)[-limit:]:
                row = json.loads(line)
                out.append({"input": row["input"], "output": row["output"]})
    return out

def extract_sql_from_reply(reply):
    reply = (reply or "").strip()
    for start in ("SELECT", "select", "INSERT", "UPDATE", "DELETE", "WITH"):
        i = reply.upper().find(start.upper())
        if i != -1:
            return reply[i:].strip()
    return reply

def normalize_sql(s):
    if not s or not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def get_content_from_nim_response(data):
    """Extract assistant text from NIM chat response. Handles content, content-as-list, and NIM reasoning/reasoning_content fields."""
    try:
        choice = data.get("choices", [{}])[0]
        msg = choice.get("message", {})
        content = msg.get("content")
        if isinstance(content, str) and content:
            return content
        if isinstance(content, list):
            parts = []
            for p in content:
                if isinstance(p, dict):
                    parts.append(p.get("text", p.get("content", "")))
                elif isinstance(p, str):
                    parts.append(p)
            return "\n".join(parts) if parts else ""
        # NIM Nemotron may put generated text in reasoning_content or reasoning when content is null
        rc = msg.get("reasoning_content") or msg.get("reasoning")
        if isinstance(rc, str) and rc.strip():
            return rc.strip()
        return ""
    except Exception:
        return ""

eval_data_7b = load_eval_data_7b(limit=EVAL_LIMIT_7B)
print(f"Eval set size: {len(eval_data_7b)} (EVAL_LIMIT_7B={EVAL_LIMIT_7B})")

url = f"{NIM_BASE_URL}/v1/chat/completions"
exact_ok = norm_ok = 0
n = len(eval_data_7b)

for i, ex in enumerate(eval_data_7b):
    payload = {
        "model": NIM_MODEL_ID,
        "messages": [
            {"role": "system", "content": NIM_SYSTEM_FOR_SQL},
            {"role": "user", "content": ex["input"].strip() + "\n\nOutput only the SQL query, nothing else."},
        ],
        "max_tokens": MAX_NEW_TOKENS_7B,
        "temperature": 0,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    try:
        r = requests.post(url, json=payload, timeout=120)
        r.raise_for_status()
        content = get_content_from_nim_response(r.json())
    except Exception as e:
        content = ""
        print(f"  Request {i+1} failed: {e}")
    pred = extract_sql_from_reply(content)
    gold = ex["output"]
    if pred.strip() == gold.strip():
        exact_ok += 1
    if normalize_sql(pred) == normalize_sql(gold):
        norm_ok += 1
    exact_pct = round(100.0 * exact_ok / (i + 1), 1)
    norm_pct = round(100.0 * norm_ok / (i + 1), 1)
    print(f"  [NIM] Eval {i+1}/{n} | exact: {exact_pct}% | normalized: {norm_pct}%", flush=True)

print("\n--- NIM Eval Results ---")
print(f"Exact match %:        {round(100.0 * exact_ok / n, 2) if n else 0}")
print(f"Normalized match %:   {round(100.0 * norm_ok / n, 2) if n else 0}")
print(f"N samples:            {n}")

Eval set size: 50 (EVAL_LIMIT_7B=50)
  [NIM] Eval 1/50 | exact: 0.0% | normalized: 0.0%
  [NIM] Eval 2/50 | exact: 0.0% | normalized: 0.0%
  [NIM] Eval 3/50 | exact: 0.0% | normalized: 0.0%
  [NIM] Eval 4/50 | exact: 0.0% | normalized: 0.0%
  [NIM] Eval 5/50 | exact: 0.0% | normalized: 0.0%
  [NIM] Eval 6/50 | exact: 0.0% | normalized: 0.0%
  [NIM] Eval 7/50 | exact: 0.0% | normalized: 0.0%
  [NIM] Eval 8/50 | exact: 0.0% | normalized: 0.0%
  [NIM] Eval 9/50 | exact: 0.0% | normalized: 0.0%
  [NIM] Eval 10/50 | exact: 0.0% | normalized: 0.0%
  [NIM] Eval 11/50 | exact: 0.0% | normalized: 0.0%
  [NIM] Eval 12/50 | exact: 0.0% | normalized: 0.0%
  [NIM] Eval 13/50 | exact: 0.0% | normalized: 0.0%
  [NIM] Eval 14/50 | exact: 0.0% | normalized: 0.0%
  [NIM] Eval 15/50 | exact: 0.0% | normalized: 0.0%
  [NIM] Eval 16/50 | exact: 0.0% | normalized: 0.0%
  [NIM] Eval 17/50 | exact: 0.0% | normalized: 0.0%
  [NIM] Eval 18/50 | exact: 0.0% | normalized: 0.0%
  [NIM] Eval 19/50 | exact: 0.0% | n